### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [5]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)


In [3]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [4]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000216277F8BF0>, search_type='mmr', search_kwargs={'k': 5})

In [6]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

llm=init_chat_model("openai:o4-mini")
llm


ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000216E025CA70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000216E1651730>, root_client=<openai.OpenAI object at 0x00000216E0163B00>, root_async_client=<openai.AsyncOpenAI object at 0x00000216E025CB00>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [7]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000216E025CA70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000216E1651730>, root_client=<openai.OpenAI object at 0x0

In [8]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'LangChain memory management OR “LangChain memory modules” OR “conversation memory” OR “context persistence” OR “stateful chat” OR “knowledge retention” OR “memory buffer” OR “memory chain” OR “RAG memory” OR “retrieval‐augmented memory”  \nAND (ConversationBufferMemory OR ConversationSummaryMemory OR EntityMemory OR CompositeMemory)  \nAND (vector store OR embedding store OR memory backend OR Redis OR Chroma OR FAISS OR Pinecone OR Weaviate)  \nAND (short-term vs long-term memory OR context window management OR memory injection OR prompt memory OR memory schema OR session persistence)  \nAND (technical terms: retrieval-augmented generation, embedding-based retrieval, context retrieval, memory caching, state management in LLM pipelines)'

In [9]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [10]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [11]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:  
“What memory and state‐management modules does LangChain support? For example, details on conversational buffer memory, summary memory, knowledge‐base or document memory, and embedding‐based memory. Which storage backends (in-memory, file/system, SQL/NoSQL databases, Redis, Pinecone, FAISS, Chroma, Annoy, Milvus, Weaviate, etc.) are available? How do the ephemeral vs. persistent memory options compare? Include technical terms such as RAG (retrieval-augmented generation), vector stores, semantic embeddings, cache layers, and memory serialization formats.”
✅ Answer:
 LangChain’s built-in “memory” modules (as of the versions cited) are:

1. ConversationBufferMemory  
   – Keeps the full history of the dialogue in a buffer.  
2. ConversationSummaryMemory  
   – Compresses past turns into an evolving summary to stay within token limits.


In [12]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:

("CrewAI agents" OR "Crew AI agents" OR "CrewAI bots" OR "AI-powered crew assistants" OR "autonomous crew agents" OR "collaborative AI agents" OR "intelligent crew bots" OR "multi-agent system")  
AND  
(features OR capabilities OR use cases OR applications OR integration OR deployment OR performance OR architecture OR API OR SDK OR system design OR reinforcement learning OR natural language processing OR agent-based modeling OR distributed AI OR task automation OR workflow optimization)
✅ Answer:
 CrewAI agents are autonomous AI “workers” that you assemble into a team, each with a clearly defined role (for example, researcher, planner, executor).  Once configured—via a simple YAML or JSON‐style spec that names each agent’s role, goals, memory, and tools—CrewAI automatically orchestrates their interactions, handles turn‐taking, and drives collective decision‐making.  By structuring agents into roles and workflows, CrewAI lets you scale both the breadth (adding more age